[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Advanced Vision-Based ADAS with Real-Time Schematic Mapping**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

In [4]:
pip install opencv-python tqdm ultralytics

^C
Note: you may need to restart the kernel to use updated packages.


## Project Overview
It is an advanced driver-assistance system (ADAS) that leverages the cutting-edge **YOLO11x** instance segmentation architecture to transform raw 4K video feeds into actionable driving intelligence. Unlike standard object detection projects, it processes environment data into a **Digital Twin Schematic**, providing a top-down radar view and real-time directional steering guidance.

The system performs three core functions simultaneously:
1.  **Multi-Class Semantic Segmentation:** High-fidelity masking of vehicles (Cars/Trucks) and road infrastructure (Lane Lines).
2.  **Dynamic Perimeter Mapping:** A custom-mapped 8-point sensor zone that monitors road curvature and lane drift.
3.  **Schematic Radar HUD:** A noise-filtered, top-down visualization that maps detections onto a digital canvas, mirroring professional-grade autonomous vehicle dashboards.

---

## Technical Implementation
* **Core Model:** YOLO11x (Extra Large) Instance Segmentation.
* **Inference Engine:** Optimized for NVIDIA T4/P100 GPUs using FP16 half-precision and 1024px image scaling.
* **Logic Layer:** * **Perspective-Aware Scaling:** Maps 2D image coordinates to a proportional 600x900 radar schematic.
    * **Centroid Directional Analysis:** Calculates the geometric center of the sensor zone to determine "Turn Left" vs "Turn Right" signals based on lane line intersection.
    * **Noise Filtering:** Uses a black-canvas rendering technique to exclude non-relevant environmental objects, ensuring a clean HUD.

---

## Real-World Applications



### 1. Level 2+ Autonomous Driving
The system acts as a "second set of eyes," providing lane-keeping assistance (LKA). By detecting when a lane line enters the custom perimeter, the system can trigger automated steering corrections or haptic driver alerts.

### 2. Digital Twin Dashboards
The schematic radar view can be integrated into high-end vehicle clusters (like those in Tesla, Rivian, or Lucid), allowing drivers to visualize the "AI's mind" and understand how the vehicle perceives its surroundings in low-visibility conditions.

### 3. Smart Infrastructure & Traffic Management
Beyond in-vehicle use, the logic can be applied to static 5G smart-city cameras to monitor traffic flow, detect lane violations, and predict potential collisions at complex intersections.

### 4. Blind Spot & Collision Avoidance
The perimeter mapping logic can be extended to the sides and rear of the vehicle to create a 360-degree safety bubble, identifying "shadow" vehicles in the radar view that might be missed by traditional mirrors.

##  Installation & Setup

To use the **Labellerr Fine-Tune Utilities** in your environment (Kaggle, Colab, or Local), you can clone the repository directly into your workspace. This allows you to access the specialized conversion scripts for your ADAS dataset.

In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [7]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['trial.mp4'],
    total_images=50,
    out_dir="input_data",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 50 frames to folder: input_data


## Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [2]:
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

coco_to_yolo_converter(
    json_path="28766079-61a3-4ab6-a4fa-941ec828bcaf.json",
    images_dir="input_data",
    output_dir="yolo_dataset",
    use_split=True,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    shuffle=True,
    verbose=True
)


Conversion complete. Stats: {'train': 70, 'val': 20, 'test': 10}


{'stats': {'train': 70, 'val': 20, 'test': 10}, 'output_dir': 'yolo_dataset'}

## Model Training Configuration

This stage of the pipeline initializes the fine-tuning process for the **YOLO11x-seg** model. By leveraging the **Extra Large (x)** variant, we prioritize maximum detection accuracy and segmentation mask smoothness, which are critical for safety-critical ADAS applications.

In [4]:
from ultralytics import YOLO

# Load the nano segmentation model
model = YOLO('yolo11n-seg.pt')

# Start training
model.train(
    data=r"C:\Users\soham\finalproject\new_pipeline\yolo_dataset\data.yaml",
    epochs=150,
    imgsz=640,
    device=0, 
    project=r'C:\Users\soham\finalproject\new_pipeline',
    name='adas_final_run'
)

Ultralytics 8.4.120  Python-3.11.15 torch-2.5.1+cu121 CUDA:0 (Quadro P1000, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\soham\finalproject\new_pipeline\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024A6724CC10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.0400

## Interactive Perimeter Mapping Tool

The **Perimeter Mapper** is a preprocessing utility designed to extract precise coordinate arrays for custom polygons. In this project, it is used to define the "sensor zone" on the road. By sampling a frame at **5.5 seconds**, we ensure the road geometry is fully visible before mapping. Essential for Lane Departure Warning Systems (LDWS). It acts as a trigger: when lane masks intersect this zone, the system calculates steering adjustments.

In [1]:
import cv2
import numpy as np

# Global list to store polygon vertices
pts = []

def draw_polygon(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        pts.append([x, y])
        print(f"Point {len(pts)} added: {x, y}")
    elif event == cv2.EVENT_RBUTTONDOWN:
        if len(pts) > 0:
            pts.pop() # Remove last point on right click
            print("Last point removed.")

# 1. Load your specific ADAS video
video_path = r'C:\Users\soham\finalproject\new_pipeline\trial.mp4'

cap = cv2.VideoCapture(video_path)

# Get FPS
fps = cap.get(cv2.CAP_PROP_FPS)

# Calculate frame number at 3 seconds
frame_number = int(fps * 5.5)

# Set video position to that frame
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

# Read frame
ret, frame = cap.read()

# Resize for your MSI screen (Mapping on 1280x720)
scale_w, scale_h = 1280, 720
frame_reshaped = cv2.resize(frame, (scale_w, scale_h))

cv2.namedWindow('Polygon Mapper')
cv2.setMouseCallback('Polygon Mapper', draw_polygon)

print("--- INSTRUCTIONS ---")
print("1. Left Click to add points (make as many as you want for a curve)")
print("2. Right Click to undo a point")
print("3. Press 's' to print the final array")
print("4. Press 'q' to quit")

while True:
    img_display = frame_reshaped.copy()
    
    # Draw the lines connecting your points
    if len(pts) > 1:
        cv2.polylines(img_display, [np.array(pts)], False, (0, 255, 255), 2)
    
    # Draw a small circle at each vertex
    for p in pts:
        cv2.circle(img_display, (p[0], p[1]), 4, (0, 0, 255), -1)
    
    cv2.imshow('Polygon Mapper', img_display)
    key = cv2.waitKey(1) & 0xFF
    
    if key == ord('s'):
        print("\n--- COPY THIS INTO YOUR MAIN SCRIPT ---")
        print(f"ZONE_POINTS = np.array({pts}, np.int32)")
        # Show closed polygon preview
        if len(pts) > 2:
            cv2.fillPoly(img_display, [np.array(pts)], (0, 255, 0))
            cv2.imshow('Polygon Mapper', img_display)
            cv2.waitKey(1000)
            
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

--- INSTRUCTIONS ---
1. Left Click to add points (make as many as you want for a curve)
2. Right Click to undo a point
3. Press 's' to print the final array
4. Press 'q' to quit
Point 1 added: (302, 715)
Point 2 added: (434, 591)
Point 3 added: (481, 536)
Point 4 added: (686, 526)
Point 5 added: (957, 719)
Point 6 added: (303, 715)

--- COPY THIS INTO YOUR MAIN SCRIPT ---
ZONE_POINTS = np.array([[302, 715], [434, 591], [481, 536], [686, 526], [957, 719], [303, 715]], np.int32)


Here is the clean Markdown code for the ADAS High-Visibility Signal System. This version is concise, structured, and ready to be added to your project documentation.

Markdown
# ADAS High-Visibility Signal System

The **Signal System** is the decision-making core of this ADAS project. It processes real-time segmentation masks to provide directional steering guidance and a high-contrast HUD for the driver.



---

## Technical Logic

* **Model:** YOLO11x-seg (Inference at 1024px with FP16 half-precision).
* **Tripwire Detection:** Utilizes `cv2.pointPolygonTest` to monitor the intersection between detected lane masks and a custom 8-point sensor zone.
* **Directional Logic:**
    * The system calculates the horizontal center ($X_{mid}$) of the sensor zone.
    * If a lane line (Class 2) enters the zone:
        * **$X < X_{mid}$:** Triggers a **"Turn Right"** signal (Orange).
        * **$X > X_{mid}$:** Triggers a **"Turn Left"** signal (Yellow).

---

## HUD & Visualization Features



* **High-Opacity Zone:** A 50% alpha-blended gray overlay highlights the active "safety bubble" on the road.
* **Color-Coded Feedback:** Uses high-contrast status colors (White for clear, Orange/Yellow for alerts) to ensure driver visibility.
* **Status HUD:** A dedicated information box in the top-right corner provides real-time system feedback using `FONT_HERSHEY_DUPLEX`.

In [39]:
import cv2
from ultralytics import YOLO
import numpy as np

# 1. Paths & Model Setup
WEIGHTS_PATH = r'best.pt'
VIDEO_PATH = r'C:\Users\soham\finalproject\new_pipeline\videoplayback.mp4'
OUTPUT_PATH = r'C:\Users\soham\finalproject\new_pipeline\output_path\final4.mp4'

# Class-to-Color Mapping (BGR format for OpenCV)
class_colors = {
    0: (0, 165, 255),      # Bus = Orange
    1: (0, 0, 255),        # Car = Red
    2: (0, 255, 0),        # Full Dashed White = Green
    3: (255, 0, 255),      # Pedestrian = Magenta
    4: (255, 0, 0),        # Truck = Blue
    5: (0, 255, 255)       # Yellow Lane = Yellow
}

class_names = {
    0: "Bus",
    1: "Car", 
    2: "Dashed Lane",
    3: "Pedestrian",
    4: "Truck",
    5: "Yellow Lane"
}

model = YOLO(WEIGHTS_PATH)
model.to('cuda')

# Class-to-Color Mapping (BGR format)
class_colors = {
    0: (0, 0, 255),        # Car = Red
    1: (0, 165, 255),      # Bus = Orange
    2: (0, 255, 0),        # Full Dashed White = Green
    3: (255, 0, 255),      # Pedestrian = Magenta
    4: (255, 0, 0),        # Truck = Blue
    5: (0, 255, 255)       # Yellow Lane = Yellow
}

class_names = {
    0: "Bus",
    1: "Car", 
    2: "Dashed Lane",
    3: "Pedestrian",
    4: "Truck",
    5: "Yellow Lane"
}
RADAR_W = 600
RADAR_H = 900
cap = cv2.VideoCapture(VIDEO_PATH)
w, h = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), int(cap.get(5)), (w, h))
# Radar scale calculation
SCALE_X_RADAR = RADAR_W / w
SCALE_Y_RADAR = RADAR_H / h

# 2. Polygon Processing
raw_pts = [[440, 430], [548, 270], [710, 268], [828, 411], [712, 398], [632, 396], [553, 405], [440, 430]]
ZONE_POINTS = np.array([[int(x * 3.0), int(y * 3.0)] for x, y in raw_pts], np.int32)

# Calculate Polygon Center X for directional logic
x_coords = ZONE_POINTS[:, 0]
poly_center_x = (np.min(x_coords) + np.max(x_coords)) // 2



frame_idx = 0

print(f"🚀 Processing full video @ {w}x{h} resolution...")

while True:
    ret, frame = cap.read()
    
    if not ret:
          print(f"✅ End of video stream at frame {frame_idx}")
          break


    results = model(frame, conf=0.15, imgsz=1024, quantize='fp16', verbose=False)[0]
    overlay = frame.copy()
    
    signal = "Path Clear"
    signal_color = (255, 255, 255) # White

    # --- 1. Draw HIGHER OPACITY Polygon Overlay ---
    # We use a brighter gray and will blend it with higher weight
    cv2.fillPoly(overlay, [ZONE_POINTS], (220, 220, 220))

    if results.masks is not None:
        classes = results.boxes.cls.cpu().numpy()
        masks = results.masks.xy
        
        for i, mask_data in enumerate(masks):
            cls = int(classes[i])
            pts = np.array(mask_data, dtype=np.int32)

            # Segmentation Colors - use dictionary lookup
            color = class_colors.get(cls, (200, 200, 200))
            
            # Render lanes as lines, vehicles as filled
            if cls in [2, 5]:  # Lane classes
                cv2.polylines(overlay, [pts], True, color, 6)
            else:  # Vehicle classes
                cv2.fillPoly(overlay, [pts], color)
                
                
                # Directional Logic for White Lines
                if cls == 2:
                    for p in pts:
                        if cv2.pointPolygonTest(ZONE_POINTS, (float(p[0]), float(p[1])), False) >= 0:
                            if p[0] < poly_center_x:
                                signal = "Turn Right"
                                signal_color = (0, 165, 255) # Orange
                            else:
                                signal = "Turn Left"
                                signal_color = (0, 255, 255) # Yellow
                            break

    # --- 2. Final HUD Assembly ---
    # Increased alpha to 0.5 for a much more visible polygon
    output_frame = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    # --- 3. LARGER Top Right Information Box ---
    box_w, box_h = 750, 160 # Increased width to prevent text overflow
    box_x, box_y = w - box_w - 70, 70
    
    # Darker background for better contrast
    sub_face = output_frame[box_y:box_y+box_h, box_x:box_x+box_w]
    black_rect = np.zeros(sub_face.shape, dtype=np.uint8)
    res = cv2.addWeighted(sub_face, 0.3, black_rect, 0.7, 1.0) # More opaque box
    output_frame[box_y:box_y+box_h, box_x:box_x+box_w] = res
    
    # Add Larger Signal Text
    # Font scale increased to 2.2 for maximum readability
    cv2.putText(output_frame, f"STATUS: {signal}", (box_x + 30, box_y + 100), 
                cv2.FONT_HERSHEY_DUPLEX, 2.2, signal_color, 5)

    out.write(output_frame)
    frame_idx += 1
    if frame_idx % 30 == 0: print(f"✅ Processed {frame_idx}/300")

cap.release()
out.release()
print(f"🏁 DONE! Check: {OUTPUT_PATH}")

🚀 Processing full video @ 640x360 resolution...
✅ Processed 30/300
✅ Processed 60/300
✅ Processed 90/300
✅ Processed 120/300
✅ Processed 150/300
✅ Processed 180/300
✅ Processed 210/300
✅ Processed 240/300
✅ Processed 270/300
✅ Processed 300/300
✅ Processed 330/300
✅ Processed 360/300
✅ Processed 390/300
✅ Processed 420/300
✅ Processed 450/300
✅ Processed 480/300
✅ Processed 510/300
✅ Processed 540/300
✅ Processed 570/300
✅ Processed 600/300
✅ Processed 630/300
✅ Processed 660/300
✅ Processed 690/300
✅ Processed 720/300
✅ Processed 750/300
✅ Processed 780/300
✅ Processed 810/300
✅ Processed 840/300
✅ Processed 870/300
✅ Processed 900/300
✅ Processed 930/300
✅ Processed 960/300
✅ Processed 990/300
✅ Processed 1020/300
✅ Processed 1050/300
✅ Processed 1080/300
✅ Processed 1110/300
✅ Processed 1140/300
✅ Processed 1170/300
✅ Processed 1200/300
✅ Processed 1230/300
✅ Processed 1260/300
✅ Processed 1290/300
✅ Processed 1320/300
✅ Processed 1350/300
✅ Processed 1380/300
✅ Processed 1410/300
✅

In [ ]:
import cv2
from ultralytics import YOLO
import numpy as np

# 1. Paths & Model Setup
WEIGHTS_PATH = r'best.pt'
VIDEO_PATH = r'C:\Users\soham\finalproject\new_pipeline\videoplayback.mp4'
OUTPUT_PATH = r'C:\Users\soham\finalproject\new_pipeline\output_path\final4.mp4'
model = YOLO(WEIGHTS_PATH)
model.to('cuda')

# 2. Polygon Processing
raw_pts = [[440, 430], [548, 270], [710, 268], [828, 411], [712, 398], [632, 396], [553, 405], [440, 430]]
ZONE_POINTS = np.array([[int(x * 3.0), int(y * 3.0)] for x, y in raw_pts], np.int32)

# Calculate Polygon Center X for directional logic
x_coords = ZONE_POINTS[:, 0]
poly_center_x = (np.min(x_coords) + np.max(x_coords)) // 2

cap = cv2.VideoCapture(VIDEO_PATH)
w, h = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), int(cap.get(5)), (w, h))

frame_idx = 0

print(f"🚀 Processing full video @ {w}x{h} resolution...")

while True:
    ret, frame = cap.read()
    
    if not ret:
          print(f"✅ End of video stream at frame {frame_idx}")
          break


    results = model(frame, conf=0.15, imgsz=1024, quantize='fp16', verbose=False)[0]
    overlay = frame.copy()
    
    signal = "Path Clear"
    signal_color = (255, 255, 255) # White

    # --- 1. Draw HIGHER OPACITY Polygon Overlay ---
    # We use a brighter gray and will blend it with higher weight
    # Draw perimeter zone (yellow outline + light blue fill)
    perimeter_overlay = overlay.copy()
    cv2.fillPoly(perimeter_overlay, [ZONE_POINTS], (100, 200, 255))  # Light blue
    cv2.polylines(overlay, [ZONE_POINTS], True, (255, 255, 0), 3)    # Yellow outline
    overlay = cv2.addWeighted(perimeter_overlay, 0.25, overlay, 0.75, 0)

    if results.masks is not None:
        classes = results.boxes.cls.cpu().numpy()
        masks = results.masks.xy
        
        for i, mask_data in enumerate(masks):
            cls = int(classes[i])
            pts = np.array(mask_data, dtype=np.int32)

            # Segmentation Colors - use dictionary lookup
            color = class_colors.get(cls, (200, 200, 200))
            
            # Render lanes as lines, vehicles as filled
            if cls in [2, 5]:  # Lane classes
                cv2.polylines(overlay, [pts], True, color, 6)
            else:  # Vehicle classes
                cv2.fillPoly(overlay, [pts], color)
                
                # Directional Logic for White Lines
                if cls == 2:
                    for p in pts:
                        if cv2.pointPolygonTest(ZONE_POINTS, (float(p[0]), float(p[1])), False) >= 0:
                            if p[0] < poly_center_x:
                                signal = "Turn Right"
                                signal_color = (0, 165, 255) # Orange
                            else:
                                signal = "Turn Left"
                                signal_color = (0, 255, 255) # Yellow
                            break

    # --- 2. Final HUD Assembly ---
    # Increased alpha to 0.5 for a much more visible polygon
    output_frame = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    # --- 3. LARGER Top Right Information Box ---
    box_w, box_h = 750, 160 # Increased width to prevent text overflow
    box_x, box_y = w - box_w - 70, 70
    
    # Darker background for better contrast
    sub_face = output_frame[box_y:box_y+box_h, box_x:box_x+box_w]
    black_rect = np.zeros(sub_face.shape, dtype=np.uint8)
    res = cv2.addWeighted(sub_face, 0.3, black_rect, 0.7, 1.0) # More opaque box
    output_frame[box_y:box_y+box_h, box_x:box_x+box_w] = res
    
    # Add Larger Signal Text
    # Font scale increased to 2.2 for maximum readability
    cv2.putText(output_frame, f"STATUS: {signal}", (box_x + 30, box_y + 100), 
                cv2.FONT_HERSHEY_DUPLEX, 2.2, signal_color, 5)

    out.write(output_frame)
    frame_idx += 1
    if frame_idx % 30 == 0: print(f"✅ Processed {frame_idx}/300")

cap.release()
out.release()
print(f"🏁 DONE! Check: {OUTPUT_PATH}")

🚀 Processing full video @ 640x360 resolution...
✅ Processed 30/300
✅ Processed 60/300
✅ Processed 90/300
✅ Processed 120/300
✅ Processed 150/300
✅ Processed 180/300
✅ Processed 210/300
✅ Processed 240/300
✅ Processed 270/300
✅ Processed 300/300
✅ Processed 330/300
✅ Processed 360/300
✅ Processed 390/300
✅ Processed 420/300
✅ Processed 450/300
✅ Processed 480/300
✅ Processed 510/300
✅ Processed 540/300
✅ Processed 570/300
✅ Processed 600/300
✅ Processed 630/300
✅ Processed 660/300
✅ Processed 690/300
✅ Processed 720/300
✅ Processed 750/300
✅ Processed 780/300
✅ Processed 810/300
✅ Processed 840/300
✅ Processed 870/300
✅ Processed 900/300
✅ Processed 930/300
✅ Processed 960/300
✅ Processed 990/300
✅ Processed 1020/300
✅ Processed 1050/300
✅ Processed 1080/300
✅ Processed 1110/300
✅ Processed 1140/300
✅ Processed 1170/300
✅ Processed 1200/300
✅ Processed 1230/300
✅ Processed 1260/300
✅ Processed 1290/300
✅ Processed 1320/300
✅ Processed 1350/300
✅ Processed 1380/300
✅ Processed 1410/300
✅

#  Schematic Radar Mapping & HUD Implementation

The final stage of the **ADAS** project introduces **Schematic Mapping**. This technique converts complex, high-resolution visual detections into a simplified, high-contrast digital "Radar" representation, mirroring the perception displays found in modern autonomous vehicles.



---

##  Technical Implementation

The system executes a multi-layer visualization strategy to ensure driver clarity:

1.  **Geometric Filtering:** The custom 8-point polygon acts as a spatial filter. Using `cv2.pointPolygonTest`, the system ignores environmental noise and focus exclusively on objects interacting with the vehicle's immediate trajectory.
2.  **Digital Twin Projection:** Instead of standard bounding boxes, the code utilizes **Instance Segmentation Masks**. These are re-rendered onto a dedicated overlay layer to create a pixel-perfect digital twin of the road.
3.  **Alpha Blending:** Using `cv2.addWeighted` at a **0.5 ratio**, the original video is desaturated. This allows the schematic elements—Green (Lanes), Red (Cars), and Blue (Trucks)—to "pop" with maximum visibility.
---

## Advantages of Schematic Mapping

| Advantage | Description |
| :--- | :--- |
| **Cognitive Load Reduction** | Strips away visual clutter (trees, signs, sky) so the driver only sees relevant obstacles. |
| **High-Contrast Alerts** | Uses glowing status colors (Orange/Yellow) that remain visible in any lighting condition. |
| **Zero-Inference HUD** | The large-scale text box provides unambiguous instructions (**"Turn Right"**) to support the visual data. |
| **Safety Zone Transparency** | Visualizing the "safety bubble" builds user trust by showing exactly what the AI is monitoring. |

In [36]:
# 1. Install Ultralytics
!pip install ultralytics -q

import cv2
from ultralytics import YOLO
import numpy as np
import os

# 2. Paths from your Kaggle Input
WEIGHTS_PATH = r'best.pt'
VIDEO_PATH = r'C:\Users\soham\finalproject\new_pipeline\videoplayback.mp4'
OUTPUT_PATH = r'C:\Users\soham\finalproject\new_pipeline\output_path\final4.mp4'

# 3. Initialize Model on Kaggle GPU
model = YOLO(WEIGHTS_PATH)
model.to('cuda')

# Class-to-Color Mapping (BGR format)
class_colors = {
    0: (0, 0, 255),        # Car = Red
    1: (0, 165, 255),      # Bus = Orange
    2: (0, 255, 0),        # Full Dashed White = Green
    3: (255, 0, 255),      # Pedestrian = Magenta
    4: (255, 0, 0),        # Truck = Blue
    5: (0, 255, 255)       # Yellow Lane = Yellow
}

class_names = {
    0: "Bus",
    1: "Car", 
    2: "Dashed Lane",
    3: "Pedestrian",
    4: "Truck",
    5: "Yellow Lane"
}

# 4. Perimeter & Radar Setup (Scaled 3x for 4K)
# ==========================================
# INTERACTIVE PERIMETER
# ==========================================

raw_pts = [
    [440, 430],
    [548, 270],
    [710, 268],
    [828, 411],
    [712, 398],
    [632, 396],
    [553, 405],
    [440, 430]
]

# Reference image = 1280 x 720
scale_x = w / 1280.0
scale_y = h / 720.0

ZONE_POINTS = np.array([
    [int(x * scale_x), int(y * scale_y)]
    for x, y in raw_pts
], dtype=np.int32)

poly_center_x = (
    np.min(ZONE_POINTS[:, 0]) +
    np.max(ZONE_POINTS[:, 0])
) // 2

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h)) 

cap = cv2.VideoCapture(VIDEO_PATH) 
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) 
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) 
fps = int(cap.get(cv2.CAP_PROP_FPS)) 
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) 
print(f"Video Resolution: {w} x {h}") 

# Enlarged Radar Dimensions 
RADAR_W, RADAR_H = 600, 900 
SCALE_X, SCALE_Y = RADAR_W / w, RADAR_H / h 
frame_idx = 0 
print(f"🚀 Kaggle Processing Started | Total Frames to Process: {total_frames}")
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video Resolution: {w} x {h}")

while cap.isOpened():
    ret, frame = cap.read()
    
    # --- REMOVED THE 300 FRAME LIMIT ---
    if not ret:
        break

    # YOLO11x Inference at 1024px
    results = model(frame, conf=0.15, imgsz=1024, quantize='fp16', verbose=False)[0]
    
    overlay = frame.copy()
    radar_view = np.zeros((RADAR_H, RADAR_W, 3), dtype=np.uint8)
    signal = "Path Clear"
    signal_color = (255, 255, 255)

    perimeter_overlay = overlay.copy()
    cv2.fillPoly(perimeter_overlay, [ZONE_POINTS], (100, 200, 255))  # Light blue fill
    cv2.polylines(overlay, [ZONE_POINTS], True, (255, 255, 0), 3)    # Yellow outline
    overlay = cv2.addWeighted(perimeter_overlay, 0.25, overlay, 0.75, 0)

    if results.masks is not None:
        classes = results.boxes.cls.cpu().numpy()
        masks = results.masks.xy
        
        for i, mask_data in enumerate(masks):
            cls = int(classes[i])
            pts = np.array(mask_data, dtype=np.int32)

            # Segmentation Colors
            # Segmentation Colors - use dictionary lookup
            color = class_colors.get(cls, (200, 200, 200))
            
            # Render lanes as lines, vehicles as filled
            if cls in [2, 5]:  # Lane classes
                cv2.polylines(overlay, [pts], True, color, 6)
            else:  # Vehicle classes
                cv2.fillPoly(overlay, [pts], color)
            
            # Radar Mapping
            # Radar Mapping
            # Radar Mapping
            radar_pts = (pts * [SCALE_X, SCALE_Y]).astype(np.int32)
            radar_color = class_colors.get(cls, (200, 200, 200))
            
            if cls in [2, 5]:  # Lane classes
                cv2.polylines(radar_view, [radar_pts], True, radar_color, 4)
            else:  # Vehicle classes
                cv2.fillPoly(radar_view, [radar_pts], radar_color)
            
            # Steering Logic
            if cls == 2 or cls == 5: # White Lane or Yellow Boundary
                for p in pts:
                    if cv2.pointPolygonTest(ZONE_POINTS, (float(p[0]), float(p[1])), False) >= 0:
                        if p[0] < poly_center_x:
                            signal = "Turn Right"; signal_color = (0, 165, 255)
                        else:
                            signal = "Turn Left"; signal_color = (0, 255, 255)
                        break

                        # Add class label at centroid
            M = cv2.moments(pts)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                label = class_names.get(cls, "Unknown")
                cv2.putText(overlay, label, (cx-25, cy), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Blend and Assemble HUD
    output_frame = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    # Radar View (Top Left)
    # ========== RADAR VIEW (Top Left) ==========
    radar_x, radar_y = 20, 20
    radar_w_display, radar_h_display = 130, 210
    
    radar_resized = cv2.resize(radar_view, (radar_w_display, radar_h_display))
    
    # Draw white border around radar
    cv2.rectangle(output_frame, (radar_x-4, radar_y-4), 
                  (radar_x+radar_w_display+4, radar_y+radar_h_display+4), 
                  (255, 255, 255), 3)
    
    # Draw car indicator (blue triangle) at center bottom
    car_tri = np.array([
        [radar_w_display//2, radar_h_display-15], 
        [radar_w_display//2-12, radar_h_display-35], 
        [radar_w_display//2+12, radar_h_display-35]
    ], np.int32)
    cv2.drawContours(radar_resized, [car_tri], 0, (255, 0, 0), -1)
    
    # Place radar on frame
    output_frame[radar_y:radar_y+radar_h_display, radar_x:radar_x+radar_w_display] = radar_resized

    # ========== STATUS BOX (Top Right) ==========
    status_w, status_h = 280, 50 
    status_x = w - status_w - 20 
    status_y = 20
    
    # Ensure bounds are valid
    if status_x + status_w > w:
        status_x = w - status_w - 20
    if status_y + status_h > h:
        status_y = 20
    
    try: 
        sub_face = output_frame[status_y:status_y+status_h, status_x:status_x+status_w].copy()
        black_rect = np.zeros(sub_face.shape, dtype=np.uint8) 
        res = cv2.addWeighted(sub_face, 0.65, black_rect, 0.35, 0) 
        output_frame[status_y:status_y+status_h, status_x:status_x+status_w] = res 
    except: 
        pass
    
    # Yellow border around status
    cv2.rectangle(output_frame, (status_x-3, status_y-3), 
                  (status_x+status_w+3, status_y+status_h+3), 
                  (0, 255, 255), 3)
    
    # Status text
    text = f"STATUS: {signal}"
    font = cv2.FONT_HERSHEY_DUPLEX
    font_scale = 0.85
    thickness = 2
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    text_x = status_x + (status_w - text_size[0]) // 2  # Center horizontally
    text_y = status_y + (status_h + text_size[1]) // 2  # Center vertically
    cv2.putText(output_frame, text, (text_x, text_y), font, font_scale, signal_color, thickness)

    out.write(output_frame)
    frame_idx += 1
    
    # Progress update every 100 frames
    if frame_idx % 100 == 0: 
        print(f"✅ Progress: {frame_idx}/{total_frames} frames processed...")

cap.release()
out.release()
print(f"🏁 PROJECT COMPLETE! Final video saved to: {OUTPUT_PATH}")

Video Resolution: 640 x 360
🚀 Kaggle Processing Started | Total Frames to Process: 3026
Video Resolution: 640 x 360
✅ Progress: 100/3026 frames processed...
✅ Progress: 200/3026 frames processed...
✅ Progress: 300/3026 frames processed...
✅ Progress: 400/3026 frames processed...
✅ Progress: 500/3026 frames processed...
✅ Progress: 600/3026 frames processed...
✅ Progress: 700/3026 frames processed...
✅ Progress: 800/3026 frames processed...
✅ Progress: 900/3026 frames processed...
✅ Progress: 1000/3026 frames processed...
✅ Progress: 1100/3026 frames processed...
✅ Progress: 1200/3026 frames processed...
✅ Progress: 1300/3026 frames processed...
✅ Progress: 1400/3026 frames processed...
✅ Progress: 1500/3026 frames processed...
✅ Progress: 1600/3026 frames processed...
✅ Progress: 1700/3026 frames processed...
✅ Progress: 1800/3026 frames processed...
✅ Progress: 1900/3026 frames processed...
✅ Progress: 2000/3026 frames processed...
✅ Progress: 2100/3026 frames processed...
✅ Progress:

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*


In [21]:
import json
import os
import copy

JSON_PATH = "28766079-61a3-4ab6-a4fa-941ec828bcaf.json"
FIXED_JSON_PATH = "labellerr_fixed.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------------------------------------------------------
# Find the Pedestrian category
# ---------------------------------------------------------
pedestrian_category = None

for category in data["categories"]:
    if category["name"].lower() == "pedestrian":
        pedestrian_category = category
        break

if pedestrian_category is None:
    raise ValueError("Pedestrian category not found!")

pedestrian_id = pedestrian_category["id"]
pedestrian_question_id = pedestrian_category["labellerr_question_id"]

print("Pedestrian category ID:", pedestrian_id)
print("Pedestrian question ID:", pedestrian_question_id)

# ---------------------------------------------------------
# Convert Pedestrian bounding boxes to polygons
# ---------------------------------------------------------
pedestrian_count = 0

for ann in data["annotations"]:

    # Identify actual Pedestrian annotations
    if ann.get("labelerr_question_id") == pedestrian_question_id:

        bbox = ann.get("bbox")

        if bbox is None or len(bbox) != 4:
            continue

        x, y, w, h = bbox

        # Convert bbox -> rectangular polygon
        polygon = [
            x,     y,
            x + w, y,
            x + w, y + h,
            x,     y + h
        ]

        ann["segmentation"] = [polygon]

        # Area
        ann["area"] = w * h

        pedestrian_count += 1

print("Pedestrian annotations converted:", pedestrian_count)

# ---------------------------------------------------------
# Save new JSON
# ---------------------------------------------------------
with open(FIXED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print("Fixed JSON saved as:", FIXED_JSON_PATH)

Pedestrian category ID: 3
Pedestrian question ID: c2838ad3-0f47-497a-8127-efa9a71799a7
Pedestrian annotations converted: 0
Fixed JSON saved as: labellerr_fixed.json


import os
folder_path = r"C:\Users\soham\finalproject\new_pipeline\adas_final_run" 
files = os.listdir(folder_path) print("Files inside folder:", files)

In [16]:
import os

parent_folder = r"C:\Users\soham\finalproject\new_pipeline"
print("Files in main pipeline folder:")
for f in os.listdir(parent_folder):
    if f.endswith(('.mp4', '.avi', '.mkv')):
        print("  ->", f)

Files in main pipeline folder:
  -> trial.mp4
  -> videoplayback.mp4
